# **Hypothesis Testing**

The goal of this file is to determine the effect a home run has on price in the Los Angeles Angels (LAA) and Los Angeles Dodgers (LAD) sports trading markets.

## Data Description

### Dependent Variables

- **price_at_hr**: The price of a trade at the time a home run was hit, where all prices represent the probability of the home team winning (used only in regression models with generic and forward data).

- **d_price_w $\alpha$ _b $\beta$**: The double binned mean price at the time of a home run.
    - $\alpha$ is the window size: $\alpha \in \{1, 2, 3, 4, 5, 6\}$
    - $\beta$ is the bin size: $\beta \in \{1, 2, 3, 4, 5, 6\}$

- **laa_homerun_dummy**: Binary indicator equal to 1 if LAA hit a home run at that timestamp, 0 otherwise.

- **lad_homerun_dummy**: Binary indicator equal to 1 if LAD hit a home run at that timestamp, 0 otherwise.

- **runners_on_base**: The number of runners on base when the home run was hit.

- **inning**: The inning in which the home run was hit.

- **laa_score**: LAA's score when the home run was hit.

- **lad_score**: LAD's score when the home run was hit.

- **opp_score**: The opponent's score when the home run was hit.

### Independent Variables

- **g_pct_px_chg**: The generic percent price change measuring market reaction to a home run (used only in regression models with generic data).

- **f_pct_px_chg_w $\alpha$ _b $\beta$**: The forward binned mean percent price change measuring market reaction to a home run (used only in regression models with forward binned data).

    - $\alpha$ is the window size: $\alpha \in \{1, 2, 3, 4, 5, 6\}$
    - $\beta$ is the bin size: $\beta \in \{1, 2, 3, 4, 5, 6\}$

- **d_pct_px_chg_w $\alpha$ _b $\beta$**: The double binned mean percent price change measuring market reaction to a home run (used only in regression models with double binned data).

    - $\alpha$ is the window size: $\alpha \in \{1, 2, 3, 4, 5, 6\}$
    - $\beta$ is the bin size: $\beta \in \{1, 2, 3, 4, 5, 6\}$

In [1]:
import pandas as pd
from utils.hypothesis_testing_helpers import get_model_stats, graph_model_stats

In [2]:
# Load in feature engineered data frames

laa_generic_df = pd.read_parquet("../data/processed/laa_generic_data.parquet")
laa_forward_df = pd.read_parquet("../data/processed/laa_forward_data.parquet")
laa_double_df = pd.read_parquet("../data/processed/laa_double_data.parquet")

lad_generic_df = pd.read_parquet("../data/processed/lad_generic_data.parquet")
lad_forward_df = pd.read_parquet("../data/processed/lad_forward_data.parquet")
lad_double_df = pd.read_parquet("../data/processed/lad_double_data.parquet")

In [3]:
# Rename price column

laa_generic_df = laa_generic_df.rename(columns={'price': 'price_at_hr'})
laa_forward_df = laa_forward_df.rename(columns={'price': 'price_at_hr'})

lad_generic_df = lad_generic_df.rename(columns={'price': 'price_at_hr'})
lad_forward_df = lad_forward_df.rename(columns={'price': 'price_at_hr'})

## Generic Target Regression Analysis

In [4]:
# Define features and targets

laa_generic_features = [
    'laa_homerun_dummy', 'price_at_hr', 'runners_on_base', 
    'inning', 'laa_score', 'opp_score'
]
lad_generic_features = [
    'lad_homerun_dummy', 'price_at_hr', 'runners_on_base', 
    'inning', 'lad_score', 'opp_score'
]

generic_targets = ['g_pct_px_chg']

# Look at a subset of the features, and drop any price change data if missing
laa_generic_df = laa_generic_df[generic_targets + laa_generic_features].dropna(subset=generic_targets)
lad_generic_df = lad_generic_df[generic_targets + lad_generic_features].dropna(subset=generic_targets)

laa_generic_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 328 entries, 0 to 350
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   g_pct_px_chg       328 non-null    float64
 1   laa_homerun_dummy  328 non-null    int64  
 2   price_at_hr        328 non-null    float64
 3   runners_on_base    328 non-null    float64
 4   inning             328 non-null    float64
 5   laa_score          328 non-null    float64
 6   opp_score          328 non-null    float64
dtypes: float64(6), int64(1)
memory usage: 20.5 KB


In [5]:
lad_generic_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 337 entries, 0 to 346
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   g_pct_px_chg       337 non-null    float64
 1   lad_homerun_dummy  337 non-null    int64  
 2   price_at_hr        337 non-null    float64
 3   runners_on_base    337 non-null    float64
 4   inning             337 non-null    float64
 5   lad_score          337 non-null    float64
 6   opp_score          337 non-null    float64
dtypes: float64(6), int64(1)
memory usage: 21.1 KB


### Model 1

$$\mathbb{E}[Y \mid X_1] = \beta_0 + \beta_1 X_1$$

where:

- $Y$ is **g_pct_px_chg**, and

- $X_1$ is **laa_homerun_dummy** or **lad_homerun_dummy**.

In [6]:
# Regression modeling

laa_m1_stats = get_model_stats(laa_generic_df, 'generic', generic_targets, ['laa_homerun_dummy'])
lad_m1_stats = get_model_stats(lad_generic_df, 'generic', generic_targets, ['lad_homerun_dummy'])

In [7]:
laa_m1_stats[0]

{'r_squared': 0.302969228358304,
 'beta_const': -0.17958346554007307,
 'beta_laa_homerun_dummy': 0.3488233801234973,
 'p_value_const': 8.966600673576002e-16,
 'p_value_laa_homerun_dummy': 2.2458051052433852e-27}

In [8]:
lad_m1_stats[0]

{'r_squared': 7.404495596419203e-05,
 'beta_const': 0.1399894226138077,
 'beta_lad_homerun_dummy': -0.03132184773600571,
 'p_value_const': 0.35043881570183033,
 'p_value_lad_homerun_dummy': 0.8749440770851188}

### Model 2

$$\mathbb{E}[Y \mid X_1, X_2, X_3, X_4, X_5, X_6] = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \beta_3 X_3 + \beta_4 X_4 + \beta_5 X_5 + \beta_6 X_6$$

where:

- $Y$ is **g_pct_px_chg**,

- $X_1$ is **laa_homerun_dummy** or **lad_homerun_dummy**,

- $X_2$ is the **price_at_hr**,

- $X_3$ is the **runners_on_base**,

- $X_4$ is the **inning**,

- $X_5$ is the **laa_score**, and

- $X_6$ is the **opp_score**.

In [9]:
# Regression modeling

laa_m2_stats = get_model_stats(
    laa_generic_df,
    'generic',
    generic_targets,
    laa_generic_features
)

lad_m2_stats = get_model_stats(
    lad_generic_df,
    'generic',
    generic_targets,
    lad_generic_features
)

In [10]:
laa_m2_stats[0]

{'r_squared': 0.36814149716896727,
 'beta_const': -0.011434540064995294,
 'beta_laa_homerun_dummy': 0.40922677284466014,
 'beta_price_at_hr': -0.5015138529270443,
 'beta_runners_on_base': 0.02828786002785159,
 'beta_inning': 0.003304351529895411,
 'beta_laa_score': 0.039975306635454094,
 'beta_opp_score': -0.03482554126276892,
 'p_value_const': 0.8212616278843469,
 'p_value_laa_homerun_dummy': 1.06229514260173e-31,
 'p_value_price_at_hr': 3.256058736621122e-07,
 'p_value_runners_on_base': 0.1378515722398899,
 'p_value_inning': 0.6598116163654989,
 'p_value_laa_score': 0.0019271142924141508,
 'p_value_opp_score': 0.003894494729302021}

In [11]:
lad_m2_stats[0]

{'r_squared': 0.09730609344183772,
 'beta_const': 1.6438040725113987,
 'beta_lad_homerun_dummy': 0.46086089493831917,
 'beta_price_at_hr': -3.3533166353762685,
 'beta_runners_on_base': 0.20693086868310645,
 'beta_inning': 0.032219889384368025,
 'beta_lad_score': 0.26713406820379043,
 'beta_opp_score': -0.33025710516555673,
 'p_value_const': 0.00013127826136651187,
 'p_value_lad_homerun_dummy': 0.02751009473136733,
 'p_value_price_at_hr': 1.0058702599963182e-07,
 'p_value_runners_on_base': 0.09310891851407804,
 'p_value_inning': 0.4821879305848227,
 'p_value_lad_score': 4.080248842640093e-05,
 'p_value_opp_score': 1.2564836453812656e-05}

## Forward Target Regression Analysis

In [12]:
# Define features and targets

laa_forward_features = [
    'laa_homerun_dummy', 'price_at_hr', 'runners_on_base', 
    'inning', 'laa_score', 'opp_score'
]
lad_forward_features = [
    'lad_homerun_dummy', 'price_at_hr', 'runners_on_base', 
    'inning', 'lad_score', 'opp_score'
]

forward_targets = [col for col in laa_forward_df.columns if 'f_pct_px_chg' in col]

# Look at a subset of the features, and drop any price change data if missing
laa_forward_df = laa_forward_df[forward_targets + laa_forward_features].dropna(subset=forward_targets)
lad_forward_df = lad_forward_df[forward_targets + lad_forward_features].dropna(subset=forward_targets)

laa_forward_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 305 entries, 0 to 351
Data columns (total 42 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   f_pct_px_chg_w1_b1  305 non-null    float64
 1   f_pct_px_chg_w1_b2  305 non-null    float64
 2   f_pct_px_chg_w1_b3  305 non-null    float64
 3   f_pct_px_chg_w1_b4  305 non-null    float64
 4   f_pct_px_chg_w1_b5  305 non-null    float64
 5   f_pct_px_chg_w1_b6  305 non-null    float64
 6   f_pct_px_chg_w2_b1  305 non-null    float64
 7   f_pct_px_chg_w2_b2  305 non-null    float64
 8   f_pct_px_chg_w2_b3  305 non-null    float64
 9   f_pct_px_chg_w2_b4  305 non-null    float64
 10  f_pct_px_chg_w2_b5  305 non-null    float64
 11  f_pct_px_chg_w2_b6  305 non-null    float64
 12  f_pct_px_chg_w3_b1  305 non-null    float64
 13  f_pct_px_chg_w3_b2  305 non-null    float64
 14  f_pct_px_chg_w3_b3  305 non-null    float64
 15  f_pct_px_chg_w3_b4  305 non-null    float64
 16  f_pct_px_chg_

In [13]:
lad_forward_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 309 entries, 0 to 346
Data columns (total 42 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   f_pct_px_chg_w1_b1  309 non-null    float64
 1   f_pct_px_chg_w1_b2  309 non-null    float64
 2   f_pct_px_chg_w1_b3  309 non-null    float64
 3   f_pct_px_chg_w1_b4  309 non-null    float64
 4   f_pct_px_chg_w1_b5  309 non-null    float64
 5   f_pct_px_chg_w1_b6  309 non-null    float64
 6   f_pct_px_chg_w2_b1  309 non-null    float64
 7   f_pct_px_chg_w2_b2  309 non-null    float64
 8   f_pct_px_chg_w2_b3  309 non-null    float64
 9   f_pct_px_chg_w2_b4  309 non-null    float64
 10  f_pct_px_chg_w2_b5  309 non-null    float64
 11  f_pct_px_chg_w2_b6  309 non-null    float64
 12  f_pct_px_chg_w3_b1  309 non-null    float64
 13  f_pct_px_chg_w3_b2  309 non-null    float64
 14  f_pct_px_chg_w3_b3  309 non-null    float64
 15  f_pct_px_chg_w3_b4  309 non-null    float64
 16  f_pct_px_chg_

### Model 1

$$\mathbb{E}[Y \mid X_1] = \beta_0 + \beta_1 X_1$$

where:

- $Y$ is **f_pct_px_chg_w $\alpha$ _b $\beta$**, and

- $X_1$ is **laa_homerun_dummy** or **lad_homerun_dummy**.

In [14]:
# Regression modeling

laa_m1_stats = get_model_stats(laa_forward_df, 'forward', forward_targets, ['laa_homerun_dummy'])
lad_m1_stats = get_model_stats(lad_forward_df, 'forward', forward_targets, ['lad_homerun_dummy'])

laa_m1_stats_df = pd.DataFrame(laa_m1_stats)
lad_m1_stats_df = pd.DataFrame(lad_m1_stats)

In [15]:
# Plot home run beta for LAA

fig = graph_model_stats(
    laa_m1_stats_df,
    stat_type='beta',
    title='Model 1 Coefficient',
    z='laa_homerun_dummy',
    gradient='p_value',
    team='LAA'
)

fig.show()

In [16]:
# Plot home run beta for LAD

fig = graph_model_stats(
    lad_m1_stats_df,
    stat_type='beta',
    title='Model 1 Coefficient',
    z='lad_homerun_dummy',
    gradient='p_value',
    team='LAD'
)

fig.show()

In [17]:
# Plot R² for LAA

fig = graph_model_stats(
    df = laa_m1_stats_df,
    stat_type='r_squared',
    title='Model 1 R²',
    z='laa_homerun_dummy',
    gradient='r_squared',
    team='LAA'
)

fig.show()

In [18]:
# Plot R² for LAD

fig = graph_model_stats(
    df = lad_m1_stats_df,
    stat_type='r_squared',
    title='Model 1 R²',
    z='lad_homerun_dummy',
    gradient='r_squared',
    team='LAD'
)

fig.show()

### Model 2

$$\mathbb{E}[Y \mid X_1, X_2, X_3, X_4, X_5, X_6] = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \beta_3 X_3 + \beta_4 X_4 + \beta_5 X_5 + \beta_6 X_6$$

where:

- $Y$ is **f_pct_px_chg_w $\alpha$ _b $\beta$**,

- $X_1$ is **laa_homerun_dummy** or **lad_homerun_dummy**,

- $X_2$ is the **price_at_hr**,

- $X_3$ is the **runners_on_base**,

- $X_4$ is the **inning**,

- $X_5$ is the **laa_score**, and

- $X_6$ is the **opp_score**.

In [19]:
# Regression modeling

laa_m2_stats = get_model_stats(laa_forward_df, 'forward', forward_targets, laa_forward_features)
lad_m2_stats = get_model_stats(lad_forward_df, 'forward', forward_targets, lad_forward_features)

laa_m2_stats_df = pd.DataFrame(laa_m2_stats)
lad_m2_stats_df = pd.DataFrame(lad_m2_stats)

In [20]:
# Plot home run beta for LAA

fig = graph_model_stats(
    laa_m2_stats_df,
    stat_type='beta',
    title='Model 2 Coefficient',
    z='laa_homerun_dummy',
    gradient='p_value',
    team='LAA'
)

fig.show()

In [21]:
# Plot home run beta for LAD

fig = graph_model_stats(
    lad_m2_stats_df,
    stat_type='beta',
    title='Model 2 Coefficient',
    z='lad_homerun_dummy',
    gradient='p_value',
    team='LAD'
)

fig.show()

In [22]:
# Plot R² for LAA

fig = graph_model_stats(
    df = laa_m2_stats_df,
    stat_type='r_squared',
    title='Model 2 R²',
    z='laa_homerun_dummy',
    gradient='r_squared',
    team='LAA'
)

fig.show()

In [23]:
# Plot R² for LAD

fig = graph_model_stats(
    df = lad_m2_stats_df,
    stat_type='r_squared',
    title='Model 2 R²',
    z='lad_homerun_dummy',
    gradient='r_squared',
    team='LAD'
)

fig.show()

## Double Target Regression Analysis

In [24]:
# Define features and targets

double_px_features = [col for col in laa_double_df.columns if 'd_price' in col]

laa_double_features = [
    'laa_homerun_dummy', 'runners_on_base', 
    'inning', 'laa_score', 'opp_score'
] + double_px_features
lad_double_features = [
    'lad_homerun_dummy', 'runners_on_base', 
    'inning', 'lad_score', 'opp_score'
] + double_px_features

double_targets = [col for col in laa_double_df.columns if 'd_pct_px_chg' in col]

# Look at a subset of the features, and drop any price change data if missing
laa_double_df = laa_double_df[double_targets + laa_double_features].dropna(subset=double_targets)
lad_double_df = lad_double_df[double_targets + lad_double_features].dropna(subset=double_targets)

laa_double_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 288 entries, 0 to 351
Data columns (total 77 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   d_pct_px_chg_w1_b1  288 non-null    float64
 1   d_pct_px_chg_w1_b2  288 non-null    float64
 2   d_pct_px_chg_w1_b3  288 non-null    float64
 3   d_pct_px_chg_w1_b4  288 non-null    float64
 4   d_pct_px_chg_w1_b5  288 non-null    float64
 5   d_pct_px_chg_w1_b6  288 non-null    float64
 6   d_pct_px_chg_w2_b1  288 non-null    float64
 7   d_pct_px_chg_w2_b2  288 non-null    float64
 8   d_pct_px_chg_w2_b3  288 non-null    float64
 9   d_pct_px_chg_w2_b4  288 non-null    float64
 10  d_pct_px_chg_w2_b5  288 non-null    float64
 11  d_pct_px_chg_w2_b6  288 non-null    float64
 12  d_pct_px_chg_w3_b1  288 non-null    float64
 13  d_pct_px_chg_w3_b2  288 non-null    float64
 14  d_pct_px_chg_w3_b3  288 non-null    float64
 15  d_pct_px_chg_w3_b4  288 non-null    float64
 16  d_pct_px_chg_

In [25]:
lad_double_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 294 entries, 0 to 346
Data columns (total 77 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   d_pct_px_chg_w1_b1  294 non-null    float64
 1   d_pct_px_chg_w1_b2  294 non-null    float64
 2   d_pct_px_chg_w1_b3  294 non-null    float64
 3   d_pct_px_chg_w1_b4  294 non-null    float64
 4   d_pct_px_chg_w1_b5  294 non-null    float64
 5   d_pct_px_chg_w1_b6  294 non-null    float64
 6   d_pct_px_chg_w2_b1  294 non-null    float64
 7   d_pct_px_chg_w2_b2  294 non-null    float64
 8   d_pct_px_chg_w2_b3  294 non-null    float64
 9   d_pct_px_chg_w2_b4  294 non-null    float64
 10  d_pct_px_chg_w2_b5  294 non-null    float64
 11  d_pct_px_chg_w2_b6  294 non-null    float64
 12  d_pct_px_chg_w3_b1  294 non-null    float64
 13  d_pct_px_chg_w3_b2  294 non-null    float64
 14  d_pct_px_chg_w3_b3  294 non-null    float64
 15  d_pct_px_chg_w3_b4  294 non-null    float64
 16  d_pct_px_chg_

### Model 1

$$\mathbb{E}[Y \mid X_1] = \beta_0 + \beta_1 X_1$$

where:

- $Y$ is **d_pct_px_chg_w $\alpha$ _b $\beta$**, and

- $X_1$ is **laa_homerun_dummy** or **lad_homerun_dummy**.

In [26]:
# Regression modeling

laa_m1_stats = get_model_stats(laa_double_df, 'double', double_targets, ['laa_homerun_dummy'])
lad_m1_stats = get_model_stats(lad_double_df, 'double', double_targets, ['lad_homerun_dummy'])

laa_m1_stats_df = pd.DataFrame(laa_m1_stats)
lad_m1_stats_df = pd.DataFrame(lad_m1_stats)

In [27]:
# Plot home run beta for LAA

fig = graph_model_stats(
    laa_m1_stats_df,
    stat_type='beta',
    title='Model 1 Coefficient',
    z='laa_homerun_dummy',
    gradient='p_value',
    team='LAA'
)

fig.show()

In [28]:
# Plot home run beta for LAD

fig = graph_model_stats(
    lad_m1_stats_df,
    stat_type='beta',
    title='Model 1 Coefficient',
    z='lad_homerun_dummy',
    gradient='p_value',
    team='LAD'
)

fig.show()

In [29]:
# Plot R² for LAA

fig = graph_model_stats(
    df = laa_m1_stats_df,
    stat_type='r_squared',
    title='Model 1 R²',
    z='laa_homerun_dummy',
    gradient='r_squared',
    team='LAA'
)

fig.show()

In [30]:
# Plot R² for LAD

fig = graph_model_stats(
    df = lad_m1_stats_df,
    stat_type='r_squared',
    title='Model 1 R²',
    z='lad_homerun_dummy',
    gradient='r_squared',
    team='LAD'
)

fig.show()

### Model 2

$$\mathbb{E}[Y \mid X_1, X_2, X_3, X_4, X_5, X_6] = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \beta_3 X_3 + \beta_4 X_4 + \beta_5 X_5 + \beta_6 X_6$$

where:

- $Y$ is **d_pct_px_chg_w $\alpha$ _b $\beta$**,

- $X_1$ is **laa_homerun_dummy** or **lad_homerun_dummy**,

- $X_2$ is the **runners_on_base**,

- $X_3$ is the **inning**,

- $X_4$ is the **laa_score**,

- $X_5$ is the **opp_score**, and

- $X_6$ is the **d_price_w $\alpha$ _b $\beta$**.

In [31]:
# Regression modeling

laa_m2_stats = get_model_stats(laa_double_df, 'double', double_targets, laa_double_features)
lad_m2_stats = get_model_stats(lad_double_df, 'double', double_targets, lad_double_features)

laa_m2_stats_df = pd.DataFrame(laa_m2_stats)
lad_m2_stats_df = pd.DataFrame(lad_m2_stats)

In [32]:
# Plot home run beta for LAA

fig = graph_model_stats(
    laa_m2_stats_df,
    stat_type='beta',
    title='Model 2 Coefficient',
    z='laa_homerun_dummy',
    gradient='p_value',
    team='LAA'
)

fig.show()

In [33]:
# Plot home run beta for LAD

fig = graph_model_stats(
    lad_m2_stats_df,
    stat_type='beta',
    title='Model 2 Coefficient',
    z='lad_homerun_dummy',
    gradient='p_value',
    team='LAD'
)

fig.show()

In [34]:
# Plot R² for LAA

fig = graph_model_stats(
    df = laa_m2_stats_df,
    stat_type='r_squared',
    title='Model 2 R²',
    z='laa_homerun_dummy',
    gradient='r_squared',
    team='LAA'
)

fig.show()

In [35]:
# Plot R² for LAD

fig = graph_model_stats(
    df = lad_m2_stats_df,
    stat_type='r_squared',
    title='Model 2 R²',
    z='lad_homerun_dummy',
    gradient='r_squared',
    team='LAD'
)

fig.show()

## Summary of Hypothesis Testing

### Generic Target
Models using the Generic method performed the worst, with the **laa_homerun_dummy** coefficient being significant across both models and the **lad_homerun_dummy** coefficient being significant for only model two.
- Model 1 has a decent $R^2$ value for LAA of 0.30 and very weak performance for LAD of 0.00.
- Model 2 has an improved $R^2$ value for LAA of 0.37 and another very weak performance for LAD of 0.10.

### Forward Target
Models using the Forward Binned Mean method performed moderately, with the **laa_homerun_dummy** coefficient being significant across both models and the **lad_homerun_dummy** coefficient being significant for only model two.
- Model 1 has a decent $R^2$ value for LAA ranging from 0.30 to 0.46 and very weak performance for LAD ranging from 0.00 to 0.00 across window bin combinations.
- Model 2 has an improved $R^2$ value for LAA ranging from 0.39 to 0.53 and weak performance for LAD ranging from 0.12 to 0.12 across window bin combinations.

### Double Target
Models using the Double Binned Mean method performed the best, with both models having significant **laa_homerun_dummy** and **lad_homerun_dummy** coefficients across several window/bin combinations.
- Model 1 has strong $R^2$ values ranging from 0.30 to 0.63 for different targets for LAA and slightly weaker performance for LAD ranging from 0.00 to 0.46.
- Model 2 has slightly stronger performance with $R^2$ values ranging from 0.39 to 0.65 for LAA and 0.12 to 0.51 for LAD.